# Data Cleaning and Type Conversion Pipeline

This notebook processes and standardizes the labeled YouTube and Reddit datasets:
- Loads the labeled datasets from `data/`
- Converts data types to appropriate formats (numeric like counts, datetime timestamps)
- Handles missing/blank values
- Verifies data quality
- Exports standardized copies back to `data/`

Paths are resolved relative to the repository root, so the notebook runs from any
working directory.

## 1. Import Libraries

In [ ]:
import csv
from pathlib import Path
import pandas as pd


def find_repo_root():
    """Walk up from the working directory to the repo root, identified by
    data/reddit_labeled.csv. Falls back to the current directory."""
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "data" / "reddit_labeled.csv").exists():
            return candidate
    return here


REPO_ROOT = find_repo_root()
DATA_DIR = REPO_ROOT / "data"
print(f"Repo root: {REPO_ROOT}")
print(f"Data dir : {DATA_DIR}")


def to_datetime_robust(series):
    """Parse a column to datetime whether it holds Unix-epoch seconds (raw
    collector output) or already-formatted datetime strings (final data)."""
    if pd.api.types.is_numeric_dtype(series):
        return pd.to_datetime(series, unit="s", errors="coerce")
    return pd.to_datetime(series, errors="coerce")

## 2. YouTube Data Processing

### 2.1 Load and Explore

In [ ]:
fname = DATA_DIR / "youtube_labeled.csv"
df_check = pd.read_csv(fname)

print("YouTube Dataset - Initial Info")
print("=" * 50)
print(f"Shape: {df_check.shape}")
print(f"\nColumns: {df_check.columns.tolist()}")
print(f"\nData Types:")
print(df_check.dtypes)

### 2.2 Convert Data Types and Handle Missing Values

In [ ]:
def convert_like_count(val):
    """Convert likeCount values with 'k' notation to actual numbers, empty strings to 0"""
    if pd.isna(val) or val == ' ' or val == '':
        return 0  # Treat empty/space as 0
    val_str = str(val).strip()
    if val_str.endswith('k') or val_str.endswith('K'):
        try:
            return float(val_str[:-1]) * 1000
        except ValueError:
            return 0
    try:
        return float(val_str)
    except ValueError:
        return 0

df_check['likeCount'] = df_check['likeCount'].apply(convert_like_count)
df_check['created_time'] = to_datetime_robust(df_check['created_time'])
df_check['video_date'] = to_datetime_robust(df_check['video_date'])
df_check['index'] = pd.to_numeric(df_check['index'], errors='coerce').astype('Int64')

print("YouTube Dataset - Converted Data Types")
print("=" * 50)
print(df_check.dtypes)

### 2.3 Verify Data Quality

In [ ]:
print("YouTube Dataset - Missing Values Check")
print("="*50)
print("\nMissing values per column:")
missing_info = df_check.isnull().sum()
print(missing_info)
print("\nPercentage of missing values per column:")
missing_percent = (df_check.isnull().sum() / len(df_check)) * 100
print(missing_percent.round(2))
print(f"\n✓ Total rows: {len(df_check):,}")
print(f"✓ Total columns: {len(df_check.columns)}")
print(f"✓ No missing values: {df_check.isnull().sum().sum() == 0}")

### 2.4 Export Cleaned Dataset

In [ ]:
fname_export = DATA_DIR / "youtube_labeled_cleaned.csv"
df_check.to_csv(fname_export, index=False)
print(f"YouTube data exported to {fname_export}")

## 3. Reddit Data Processing

### 3.1 Load and Explore

In [ ]:
reddit_fname = DATA_DIR / "reddit_labeled.csv"
df_reddit = pd.read_csv(reddit_fname)

print("Reddit Dataset - Initial Info")
print("=" * 50)
print(f"Shape: {df_reddit.shape}")
print(f"\nColumns: {df_reddit.columns.tolist()}")
print(f"\nData Types:")
print(df_reddit.dtypes)

In [ ]:
print("\nReddit Dataset - Missing Values (Before Conversion)")
print("="*50)
print("\nMissing values per column:")
missing_info = df_reddit.isnull().sum()
print(missing_info)
print("\nPercentage of missing values per column:")
missing_percent = (df_reddit.isnull().sum() / len(df_reddit)) * 100
print(missing_percent.round(2))

### 3.2 Convert Data Types

In [ ]:
# Convert timestamp columns to datetime. to_datetime_robust handles both
# Unix-epoch integers (raw collector output) and pre-formatted strings.
df_reddit['created_time'] = to_datetime_robust(df_reddit['created_time'])
df_reddit['post_created_time'] = to_datetime_robust(df_reddit['post_created_time'])

# Convert index to Int64 (nullable integer)
df_reddit['index'] = pd.to_numeric(df_reddit['index'], errors='coerce').astype('Int64')

print("Reddit Dataset - Converted Data Types")
print("=" * 50)
print(df_reddit.dtypes)

### 3.3 Verify Data Quality

In [ ]:
print("Reddit Dataset - Final Missing Values Check")
print("="*50)
print("\nMissing values per column:")
print(df_reddit.isnull().sum())
print("\nPercentage of missing values:")
missing_percent = (df_reddit.isnull().sum() / len(df_reddit)) * 100
print(missing_percent.round(2))
print(f"\n✓ Total rows: {len(df_reddit):,}")
print(f"✓ Total columns: {len(df_reddit.columns)}")
print(f"✓ No missing values: {df_reddit.isnull().sum().sum() == 0}")

### 3.4 Export Cleaned Dataset

In [ ]:
reddit_fname_export = DATA_DIR / "reddit_labeled_cleaned.csv"
df_reddit.to_csv(reddit_fname_export, index=False)
print(f"Reddit data exported to {reddit_fname_export}")

## Summary

This notebook standardizes the column types of the labeled YouTube and Reddit
datasets and verifies data quality. The type conversions are **idempotent** and
robust to both the raw collector output (Unix-epoch timestamps, `k`-notation
like counts) and the already-cleaned final data.

**YouTube dataset** — like counts converted to numeric, timestamps parsed to
datetime, exported to `data/youtube_labeled_cleaned.csv`.

**Reddit dataset** — comment/post timestamps parsed to datetime, index cast to a
nullable integer, exported to `data/reddit_labeled_cleaned.csv`.

> The canonical labeled inputs live in `data/reddit_labeled.csv` and
> `data/youtube_labeled.csv`; the exact row/column counts are printed by the
> verification cells above so this summary never drifts from the data.